# **Previsão de Churn de Clientes (Dados)**
O dataset utilizado foi o Telco Customer Churn retirado o Kaggle.
Achei ele um bom dataset pra treinar já que possui a anatomia necessária pra prática de tratamento de dados (Possui dados que vão precisar de escalonamento, uso do OneHotEncoder, preenchimento de valores com o Imputer e etc...)

In [27]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## **Separando as colunas numericas das categóricas e do rótulo**

In [9]:
id_col = "customerID"
colunas_numericas = ["tenure", "MonthlyCharges", "TotalCharges"]
target = "Churn"
colunas_categoricas = ["gender", "SeniorCitizen", "Partner", "Dependents",
                        "PhoneService", "MultipleLines", "InternetService",
                        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                        "TechSupport", "StreamingTV", "StreamingMovies",
                        "Contract", "PaperlessBilling", "PaymentMethod"]

Usando o OneHotEncoder pra criar uma coluna para cada valor das colunas categóricas e criando o dataframe final

In [11]:
categoria_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="if_binary")
categoria_encoder.fit(df[colunas_categoricas])
encoded_categorias = categoria_encoder.transform(df[colunas_categoricas])
encoded_df = pd.DataFrame(
    encoded_categorias,
    columns=categoria_encoder.get_feature_names_out(colunas_categoricas),
    index=df.index
)

df_final = pd.concat([df[colunas_numericas], encoded_df, df[target]], axis=1)

Ao checar o tipo da coluna **TotalCharges** descobri que havia linhas vazias ja que a coluna numérica estava com rotulo de Objeto

In [24]:
df["TotalCharges"].dtype

dtype('O')

Em seguida procurei a quantidade de colunas pra avaliar se era melhor descarta-las ou preencher-las. Mesmo sendo apenas 11 de uma dataset de 7000 linhas preferi preencher com a media dos valores utilizando o imputer

In [25]:
(df["TotalCharges"].str.strip() == "").sum()

np.int64(11)

In [26]:
df_final["TotalCharges"] = pd.to_numeric(df_final["TotalCharges"], errors="coerce")

imputer = SimpleImputer(strategy="median")
df_final["TotalCharges"] = imputer.fit_transform(df_final[["TotalCharges"]])
df_final["TotalCharges"].dtype

dtype('float64')

## **Criando o conjunto de treino e de testes**

In [29]:
X = df_final.drop(columns=["Churn"])
y = df_final["Churn"]

X_treino, X_test, y_treino, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

## **Tratando as colunas com valores discrepantes**

Precisei tratar estas colunas pois os valores iam desde dezenas até milhares, o que prejudicaria o modelo


In [31]:
colunas_numericas = ["tenure", "MonthlyCharges", "TotalCharges"]

scaler = StandardScaler()
X_treino[colunas_numericas] = scaler.fit_transform(X_treino[colunas_numericas])
X_test[colunas_numericas] = scaler.transform(X_test[colunas_numericas])
X_treino[colunas_numericas].describe()

,tenure,MonthlyCharges,TotalCharges
count,5.634000e+03,5.634000e+03,5.634000e+03
mean,-1.891754e-17,-1.135052e-17,-3.783508e-18
std,1.000089e+00,1.000089e+00,1.000089e+00
min,-1.322329e+00,-1.544028e+00,-1.002135e+00
25%,-9.559779e-01,-9.711977e-01,-8.309023e-01
50%,-1.418632e-01,1.848336e-01,-3.968393e-01
75%,9.164859e-01,8.319124e-01,6.737360e-01
max,1.608483e+00,1.785939e+00,2.802714e+00


In [34]:
df.to_csv('Dados_churn.csv') #exportando tabela